# C3 — Colectare comentarii YouTube
În acest notebook colectăm un eșantion  de comentarii publice de pe YouTube.
Scopul nu este să obținem corpusul final mare, ci să înțelegem fluxul:
sursă → API → comentarii brute → fișier JSONL.
La final, fiecare student salvează propriul fișier în `data/raw/`.

## 1. Ce trebuie să avem pregătit
Avem nevoie de:
- fișier `.env` în root-ul proiectului
- cheia `YOUTUBE_API_KEY`
- un handle de canal YouTube
Exemplu în `.env`:
```text
YOUTUBE_API_KEY=cheia_ta_aici

In [1]:

from pathlib import Path
import os
import json
import requests
from datetime import datetime
from dotenv import load_dotenv

## 2. Încărcăm cheia API
Notebook-ul caută fișierul `.env` în root-ul proiectului.
Dacă cheia nu este găsită, colectarea nu poate porni.

In [2]:
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")
API_KEY = os.getenv("YOUTUBE_API_KEY")
BASE_URL = "https://www.googleapis.com/youtube/v3"
print("Root proiect:", ROOT)
print("Cheie găsită:", API_KEY is not None)

Root proiect: c:\Users\Ionela\Downloads\MCS\Proiect Inginerie AI\echochamber-project-team-5
Cheie găsită: True


## 3. Alegem canalul și numărul de videoclipuri
Fiecare student schimbă `student_id` și `handle`.
Pentru exercițiu folosim puține videoclipuri, ca să nu consumăm inutil cota API.

In [3]:
student_id = "student_02"
handle = "B1TVChannel"
max_videos = 10
max_comments_per_video = 10
output_file = ROOT / "data" / "raw" / f"{student_id}_youtube_raw.jsonl"
print(output_file)

c:\Users\Ionela\Downloads\MCS\Proiect Inginerie AI\echochamber-project-team-5\data\raw\student_02_youtube_raw.jsonl


## 4. Găsim canalul YouTube

YouTube lucrează intern cu `channel_id`, nu direct cu numele canalului.
De aceea, primul pas este să transformăm handle-ul în `channel_id`.

In [5]:
channel_response = requests.get(
    f"{BASE_URL}/channels",
    params={
        "part": "id",
        "forHandle": handle,
        "key": API_KEY
    }
)
channel_data = channel_response.json()
channel_data

{'kind': 'youtube#channelListResponse',
 'etag': 'RonwGY_bfyqIH9zxtfhxaU_lReA',
 'pageInfo': {'totalResults': 1, 'resultsPerPage': 5},
 'items': [{'kind': 'youtube#channel',
   'etag': 'i-MydfuK9Moc1bCyvLBabjzBtBw',
   'id': 'UCeqNP-Wt7YNjPyH6XhMYPxw'}]}

In [6]:
channel_id = channel_data["items"][0]["id"]
channel_id

'UCeqNP-Wt7YNjPyH6XhMYPxw'

## 5. Luăm cele mai recente videoclipuri
Acum cerem ultimele videoclipuri publicate de canal.
Pentru curs folosim doar câteva videoclipuri.

In [7]:
videos_response = requests.get(
    f"{BASE_URL}/search",
    params={
        "part": "snippet",
        "channelId": channel_id,
        "type": "video",
        "order": "date",
        "maxResults": max_videos,
        "key": API_KEY
    }
)
videos_data = videos_response.json()
videos_data["items"][0]

{'kind': 'youtube#searchResult',
 'etag': 'FmCi6nxlqHBctyftOWcKO0TJJxI',
 'id': {'kind': 'youtube#video', 'videoId': 'lGms-UN-_58'},
 'snippet': {'publishedAt': '2026-05-07T15:27:51Z',
  'channelId': 'UCeqNP-Wt7YNjPyH6XhMYPxw',
  'title': 'POLITICA ZILEI. I.FOTA: Nici CG nu a avut un val de simpatie și de susținere atat de mare ca BOLOJAN',
  'description': 'I.FOTA: „Mai rău de acum nu se poate întâmpla. BOLOJAN convinge prin ONESTITATE”/ Ieșim din PARADIGMA C.GEORGESCU ...',
  'thumbnails': {'default': {'url': 'https://i.ytimg.com/vi/lGms-UN-_58/default.jpg',
    'width': 120,
    'height': 90},
   'medium': {'url': 'https://i.ytimg.com/vi/lGms-UN-_58/mqdefault.jpg',
    'width': 320,
    'height': 180},
   'high': {'url': 'https://i.ytimg.com/vi/lGms-UN-_58/hqdefault.jpg',
    'width': 480,
    'height': 360}},
  'channelTitle': 'B1',
  'liveBroadcastContent': 'none',
  'publishTime': '2026-05-07T15:27:51Z'}}

In [8]:
videos = []
for item in videos_data["items"]:
    videos.append({
        "video_id": item["id"]["videoId"],
        "video_title": item["snippet"]["title"],
        "video_date": item["snippet"]["publishedAt"][:10]
    })
videos

[{'video_id': 'lGms-UN-_58',
  'video_title': 'POLITICA ZILEI. I.FOTA: Nici CG nu a avut un val de simpatie și de susținere atat de mare ca BOLOJAN',
  'video_date': '2026-05-07'},
 {'video_id': 'Glljhus7Mwg',
  'video_title': 'POLITICA ZILEI. BOLOJAN: NU ÎN ORICE CONDIȚII. PSD să recunoască că N-ARE SOLUȚIE. P1/3',
  'video_date': '2026-05-07'},
 {'video_id': 'FKeCVYE9cwo',
  'video_title': 'Meteo. Vremea se schimb[ brusc_Știri B1TV',
  'video_date': '2026-05-07'},
 {'video_id': 'bhjYI7gCIXg',
  'video_title': 'TALK B1 cu G.Mihai. Bolojan GREU DE UCIS/Cine este TERMINATOR?/România, BIG BROTHER,  algoritm AI P3',
  'video_date': '2026-05-07'},
 {'video_id': 'uElj_cyZ8pY',
  'video_title': 'TALK B1 cu G.Mihai. BOLOJAN s-a sucit.CE CONDIȚII PUNE/Agențiile de rating vin cu SCENARIUL BOMBĂ P2',
  'video_date': '2026-05-07'},
 {'video_id': 'QJ_P7WP0NNM',
  'video_title': 'TALK B1 cu Gabriela Mihai. CE SCENARII are pe masă șeful statului?/ULTIMUL mesaj al lui BOLOJAN P1',
  'video_date': '20

## 6. Colectăm comentariile
Pentru fiecare videoclip luăm comentariile publice ordonate după relevanță.
În acest exercițiu nu folosim paginare, deci luăm maximum 100 comentarii per videoclip.

In [9]:
comments = []
for video in videos:
    print("Colectez:", video["video_title"][:80])
    comments_response = requests.get(
        f"{BASE_URL}/commentThreads",
        params={
            "part": "snippet",
            "videoId": video["video_id"],
            "maxResults": max_comments_per_video,
            "textFormat": "plainText",
            "order": "relevance",
            "key": API_KEY
        }
    )
    comments_data = comments_response.json()
    for comment_item in comments_data.get("items", []):
        snippet = comment_item["snippet"]["topLevelComment"]["snippet"]
        record = {
            "id": f"yt_{video['video_id']}_{comment_item['id']}",
            "source_platform": "youtube",
            "source_channel": handle,
            "text_raw": snippet["textDisplay"],
            "video_id": video["video_id"],
            "video_title": video["video_title"],
            "video_date": video["video_date"],
            "comment_date": snippet["publishedAt"][:10],
            "likes": snippet["likeCount"],
            "collected_at": datetime.utcnow().strftime("%Y-%m-%d")
        }
        comments.append(record)
len(comments)

Colectez: POLITICA ZILEI. I.FOTA: Nici CG nu a avut un val de simpatie și de susținere ata


C:\Users\Ionela\AppData\Local\Temp\ipykernel_11880\3206550858.py:28: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "collected_at": datetime.utcnow().strftime("%Y-%m-%d")


Colectez: POLITICA ZILEI. BOLOJAN: NU ÎN ORICE CONDIȚII. PSD să recunoască că N-ARE SOLUȚI
Colectez: Meteo. Vremea se schimb[ brusc_Știri B1TV
Colectez: TALK B1 cu G.Mihai. Bolojan GREU DE UCIS/Cine este TERMINATOR?/România, BIG BROT
Colectez: TALK B1 cu G.Mihai. BOLOJAN s-a sucit.CE CONDIȚII PUNE/Agențiile de rating vin c
Colectez: TALK B1 cu Gabriela Mihai. CE SCENARII are pe masă șeful statului?/ULTIMUL mesaj
Colectez: VIRUSUL MORTAL, identificat și în ROMÂNIA. A  MARINESCU, manager MATEI BALȘ, car
Colectez: 1,5 mil. euro - NEREGULI în finanțarea partidelor UE. Septimius PÂRVU, Expert Fo
Colectez: BOLOJAN, mesaj de ultimă oră. Atac dur la puciștii din partid_Știri B1TV
Colectez: NEGOCIERI DECISIVE la Cotroceni! Bolojan și Kelemen Hunor, față în față cu Nicuș


75

# Explorare si curatare

## 7. Inspectăm primele comentarii
Înainte să salvăm fișierul, verificăm dacă datele arată cum trebuie.

In [10]:
comments[:3]

[{'id': 'yt_lGms-UN-_58_UgwOSx75tetfq2XJFu94AaABAg',
  'source_platform': 'youtube',
  'source_channel': 'B1TVChannel',
  'text_raw': 'Bolojan e un super lider',
  'video_id': 'lGms-UN-_58',
  'video_title': 'POLITICA ZILEI. I.FOTA: Nici CG nu a avut un val de simpatie și de susținere atat de mare ca BOLOJAN',
  'video_date': '2026-05-07',
  'comment_date': '2026-05-07',
  'likes': 8,
  'collected_at': '2026-05-07'},
 {'id': 'yt_lGms-UN-_58_Ugy_XmV8sSg2Qt3_P-x4AaABAg',
  'source_platform': 'youtube',
  'source_channel': 'B1TVChannel',
  'text_raw': 'Ce cauta ND sa discute cu alte persoane din partide?? Cumva sa caute tradatori gen Presoiu? Chiar e PeSeDan😡',
  'video_id': 'lGms-UN-_58',
  'video_title': 'POLITICA ZILEI. I.FOTA: Nici CG nu a avut un val de simpatie și de susținere atat de mare ca BOLOJAN',
  'video_date': '2026-05-07',
  'comment_date': '2026-05-07',
  'likes': 5,
  'collected_at': '2026-05-07'},
 {'id': 'yt_lGms-UN-_58_Ugwr2c84X40yjbUSAIR4AaABAg',
  'source_platform': 

In [11]:
comments[0].keys()

dict_keys(['id', 'source_platform', 'source_channel', 'text_raw', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'collected_at'])

## 8. Curățare minimă a textului
Acum pornim de la `text_raw` și construim o variantă curățată în câmpul `text`.
Nu schimbăm sensul comentariului. Eliminăm doar zgomot simplu: linkuri, spații inutile, texte prea scurte și duplicate.

In [12]:
import re

def clean_text(text):
    text = re.sub(r"http\S+", "", text)      # elimină linkuri
    text = re.sub(r"\s+", " ", text)         # normalizează spațiile
    return text.strip()

## 9. Aplicăm curățarea
Pentru fiecare comentariu păstrăm textul original în `text_raw` și adăugăm textul curățat în `text`.

In [13]:
for comment in comments:
    comment["text"] = clean_text(comment["text_raw"])

comments[0]

{'id': 'yt_lGms-UN-_58_UgwOSx75tetfq2XJFu94AaABAg',
 'source_platform': 'youtube',
 'source_channel': 'B1TVChannel',
 'text_raw': 'Bolojan e un super lider',
 'video_id': 'lGms-UN-_58',
 'video_title': 'POLITICA ZILEI. I.FOTA: Nici CG nu a avut un val de simpatie și de susținere atat de mare ca BOLOJAN',
 'video_date': '2026-05-07',
 'comment_date': '2026-05-07',
 'likes': 8,
 'collected_at': '2026-05-07',
 'text': 'Bolojan e un super lider'}

## 10. Filtrăm comentariile prea scurte
Pentru exercițiu păstrăm doar comentariile care au cel puțin 60 de caractere.
Comentariile foarte scurte sunt greu de interpretat în analiza discursivă.

In [14]:
MIN_CHARS = 60

comments_clean = [
    comment for comment in comments
    if len(comment["text"]) >= MIN_CHARS
]

print("Comentarii brute:", len(comments))
print("Comentarii după filtrarea lungimii:", len(comments_clean))

Comentarii brute: 75
Comentarii după filtrarea lungimii: 48


## 11. Filtrăm textele cu prea puține litere
Comentariile formate mai ales din emoji, simboluri sau caractere izolate produc zgomot.
Păstrăm comentariile în care cel puțin 50% dintre caractere sunt litere.

In [15]:
MIN_ALPHA = 0.5

def alpha_ratio(text):
    if len(text) == 0:
        return 0
    letters = sum(char.isalpha() for char in text)
    return letters / len(text)

comments_clean = [
    comment for comment in comments_clean
    if alpha_ratio(comment["text"]) >= MIN_ALPHA
]

print("Comentarii după filtrarea literelor:", len(comments_clean))

Comentarii după filtrarea literelor: 48


## 12. Eliminăm duplicatele
Dacă același text apare de mai multe ori, îl păstrăm o singură dată.

In [16]:
seen_texts = set()
unique_comments = []

for comment in comments_clean:
    text = comment["text"].lower()
    if text not in seen_texts:
        unique_comments.append(comment)
        seen_texts.add(text)

comments_clean = unique_comments

print("Comentarii finale după deduplicare:", len(comments_clean))

Comentarii finale după deduplicare: 48


## 14. Salvăm fișierul curățat
Salvăm rezultatul în `data/cleaned/`.

In [17]:
clean_output_file = ROOT / "data" / "cleaned" / f"{student_id}_youtube_clean.jsonl"
clean_output_file.parent.mkdir(parents=True, exist_ok=True)

with clean_output_file.open("w", encoding="utf-8") as f:
    for comment in comments_clean:
        f.write(json.dumps(comment, ensure_ascii=False) + "\n")

print("Comentarii curate salvate:", len(comments_clean))
print("Fișier:", clean_output_file)

Comentarii curate salvate: 48
Fișier: c:\Users\Ionela\Downloads\MCS\Proiect Inginerie AI\echochamber-project-team-5\data\cleaned\student_02_youtube_clean.jsonl


In [18]:
with output_file.open("w", encoding="utf-8") as f:
    for comment in comments:
        f.write(json.dumps(comment, ensure_ascii=False) + "\n")

print("Comentarii curate salvate:", len(comments_clean))
print("Fisier:", clean_output_file)

Comentarii curate salvate: 48
Fisier: c:\Users\Ionela\Downloads\MCS\Proiect Inginerie AI\echochamber-project-team-5\data\cleaned\student_02_youtube_clean.jsonl


# Functia de curatare

In [17]:
import re

def clean_comments(comments, min_chars=60, min_alpha=0.5):
    cleaned = []
    seen_texts = set()
    
    for comment in comments:
        # 1. Curățare text
        text = comment["text_raw"]
        text = re.sub(r"http\S+", "", text)
        text = re.sub(r"\s+", " ", text).strip()
        
        # 2. Filtru lungime
        if len(text) < min_chars:
            continue
        
        # 3. Filtru proporție litere
        letters = sum(char.isalpha() for char in text)
        alpha_ratio = letters / len(text) if len(text) > 0 else 0
        
        if alpha_ratio < min_alpha:
            continue
        
        # 4. Filtru duplicate
        text_key = text.lower()
        if text_key in seen_texts:
            continue
        
        seen_texts.add(text_key)
        
        # 5. Păstrăm comentariul și adăugăm textul curățat
        new_comment = comment.copy()
        new_comment["text"] = text
        new_comment["lang"] = "ro"
        cleaned.append(new_comment)
    
    return cleaned

In [18]:
comments_clean = clean_comments(
    comments,
    min_chars=60,
    min_alpha=0.5
)

print("Comentarii brute:", len(comments))
print("Comentarii curate:", len(comments_clean))

Comentarii brute: 89
Comentarii curate: 53


In [19]:
for comment in comments_clean[:3]:
    print("RAW:", comment["text_raw"])
    print("CLEAN:", comment["text"])
    print("---")

RAW: Parșivul grindeanu sa nu mai bâzâie....la munca nu la întins mana
CLEAN: Parșivul grindeanu sa nu mai bâzâie....la munca nu la întins mana
---
RAW: Taci ca vati bătut joc de tara gunoiule a fost prim-ministru cea făcut au furat toți ca ala cu stuful
CLEAN: Taci ca vati bătut joc de tara gunoiule a fost prim-ministru cea făcut au furat toți ca ala cu stuful
---
RAW: Gustere Grindeanu  îți e frică să mergi la anticipate ca îți moare PSD ul duceți-vă
CLEAN: Gustere Grindeanu îți e frică să mergi la anticipate ca îți moare PSD ul duceți-vă
---


In [20]:
clean_output_file = ROOT / "data" / "cleaned" / f"{student_id}_youtube_clean.jsonl"
clean_output_file.parent.mkdir(parents=True, exist_ok=True)

with clean_output_file.open("w", encoding="utf-8") as f:
    for comment in comments_clean:
        f.write(json.dumps(comment, ensure_ascii=False) + "\n")

print("Fișier salvat:", clean_output_file)
print("Comentarii salvate:", len(comments_clean))

Fișier salvat: c:\Users\Ionela\Downloads\MCS\Proiect Inginerie AI\echochamber-project-team-5\data\cleaned\student_02_youtube_clean.jsonl
Comentarii salvate: 53


15. Ce am obținut
Am produs două fișiere:
- `data/raw/student_XX_youtube_raw.jsonl` — comentarii brute
- `data/cleaned/student_XX_youtube_clean.jsonl` — comentarii curățate
Fișierul curățat va putea fi unit cu fișierele celorlalți membri ai echipei.